In [1]:
import yfinance as yf
import plotly.graph_objects as go
import pandas

apple = yf.download(tickers="AAPL", period="60d", interval="1m")

[*********************100%***********************]  1 of 1 completed


1 Failed download:
['AAPL']: Exception('%ticker%: 1m data not available for startTime=1684718782 and endTime=1689902782. Only 7 days worth of 1m granularity data are allowed to be fetched per request.')


In [4]:
apple

,Open,High,Low,Close,Adj Close,Volume
Datetime,,,,,,
2023-04-25 09:30:00,165.190002,166.304993,165.050003,166.229996,166.229996,4476442
2023-04-25 09:45:00,166.240005,166.250000,165.619995,166.050003,166.050003,2376745
2023-04-25 10:00:00,166.039993,166.279999,165.309998,165.400101,165.400101,2527796
2023-04-25 10:15:00,165.408600,165.500000,165.059998,165.339996,165.339996,1684241
2023-04-25 10:30:00,165.339996,165.477905,165.059494,165.179993,165.179993,1607811
...,...,...,...,...,...,...
2023-07-20 14:45:00,193.240005,193.490005,193.210007,193.270004,193.270004,993464
2023-07-20 15:00:00,193.274994,193.449997,192.860001,192.960007,192.960007,1728742
2023-07-20 15:15:00,192.970001,193.009995,192.494995,192.544998,192.544998,1898884


In [5]:
apple["Target"] = (apple["Open"] < apple["Close"]).astype(int)

In [6]:
apple

,Open,High,Low,Close,Adj Close,Volume,Target
Datetime,,,,,,,
2023-04-25 09:30:00,165.190002,166.304993,165.050003,166.229996,166.229996,4476442,1
2023-04-25 09:45:00,166.240005,166.250000,165.619995,166.050003,166.050003,2376745,0
2023-04-25 10:00:00,166.039993,166.279999,165.309998,165.400101,165.400101,2527796,0
2023-04-25 10:15:00,165.408600,165.500000,165.059998,165.339996,165.339996,1684241,0
2023-04-25 10:30:00,165.339996,165.477905,165.059494,165.179993,165.179993,1607811,0
...,...,...,...,...,...,...,...
2023-07-20 14:45:00,193.240005,193.490005,193.210007,193.270004,193.270004,993464,1
2023-07-20 15:00:00,193.274994,193.449997,192.860001,192.960007,192.960007,1728742,0
2023-07-20 15:15:00,192.970001,193.009995,192.494995,192.544998,192.544998,1898884,0


In [8]:
# using RandomForestClassifier

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, min_samples_split=50, random_state=1)

train = apple.iloc[:-1200]
test = apple.iloc[-348:]

predictors = ["Close", "Volume", "Open", "High", "Low"]
model.fit(train[predictors], train["Target"])

RandomForestClassifier(min_samples_split=50, random_state=1)

In [9]:
from sklearn.metrics import precision_score

preds = model.predict(test[predictors])

In [10]:
import pandas as pd

preds = pd.Series(preds, index=test.index)

preds

Datetime
2023-06-30 10:30:00    0
2023-06-30 10:45:00    0
2023-06-30 11:00:00    0
2023-06-30 11:15:00    0
2023-06-30 11:30:00    0
                      ..
2023-07-20 14:45:00    0
2023-07-20 15:00:00    0
2023-07-20 15:15:00    0
2023-07-20 15:30:00    0
2023-07-20 15:45:00    0
Length: 348, dtype: int32

In [11]:
precision_score(test["Target"], preds)

C:\workspaces\candlestick\venv\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


0.0

In [ ]:
combined = pd.concat([test["Target"], preds], axis=1)

In [ ]:
def predict(train, test, predictors, model): 
    model.fit(train[predictors], train["Target"])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

In [ ]:
def backtest(data, model, predictors, start=1, step=50):
    all_predictions = []
    
    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
        
    return pd.concat(all_predictions)

In [ ]:
predictions=backtest(apple, model, predictors)

In [ ]:
predictions["Predictions"].value_counts()

In [ ]:
predictions["Predictions"].value_counts()

In [ ]:
precision_score(predictions["Target"], predictions["Predictions"])

In [ ]:
# using knn - https://blog.quantinsti.com/machine-learning-k-nearest-neighbors-knn-algorithm-python/
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

df = apple

# Predictor variables
df['Open-Close']= df.Open -df.Close
df['High-Low']  = df.High - df.Low
df =df.dropna()
X= df[['Open-Close', 'High-Low']]
X.head()

Y= np.where(df['Close'].shift(-1)>df['Close'],1,-1)

# Splitting the dataset
split_percentage = 0.7
split = int(split_percentage*len(df))

X_train = X[:split]
Y_train = Y[:split]

X_test = X[split:]
Y_test = Y[split:]

In [ ]:
from sklearn.metrics import accuracy_score

# Instantiate KNN learning model(k=15)
knn = KNeighborsClassifier(n_neighbors=20)

# fit the model
knn.fit(X_train, Y_train)

# Accuracy Score
accuracy_train = accuracy_score(Y_train, knn.predict(X_train))
accuracy_test = accuracy_score(Y_test, knn.predict(X_test))

print ('Train_data Accuracy: %.2f' %accuracy_train)
print ('Test_data Accuracy: %.2f' %accuracy_test)

In [ ]:
# Plotting graphs
import matplotlib.pyplot as plt

# Predicted Signal
df['Predicted_Signal'] = knn.predict(X)

# APPL Cumulative Returns
df['APPL_returns'] = np.log(df['Close']/df['Close'].shift(1))
Cumulative_APPL_returns = df[split:]['APPL_returns'].cumsum()*100

# Cumulative Strategy Returns 
df['Startegy_returns'] = df['APPL_returns']* df['Predicted_Signal'].shift(1)
Cumulative_Strategy_returns = df[split:]['Startegy_returns'].cumsum()*100

# Plot the results to visualize the performance

plt.figure(figsize=(10,5))
plt.plot(Cumulative_APPL_returns, color='r',label = 'APPL Returns')
plt.plot(Cumulative_Strategy_returns, color='g', label = 'Strategy Returns')
plt.legend()
plt.show()